# Source Coverage Benchmark

Compare one-call SourceCoverage in **serial** and **async** modes, then compare it
against a benchmark-only **DAG / two-stage** SourceCoverage approach. Tokens are not
reported here (not cleanly available yet — a separate production change).

## 1. Setup

Load the CSV and build the judge and framework (one-call `SourceCoverageEvaluator`).

In [ ]:
import time
import json
import asyncio

import pandas as pd

from idp_eval import (
    EvaluationCase,
    EvaluationFramework,
    SourceCoverageEvaluator,
    create_judge,
)

GT_COLUMN = "gt_source_coverage"  # ground-truth column, if present

df = pd.read_csv("golden_set_augmented_tagged.csv").fillna("")

judge = create_judge(verify_ssl=False)

framework = EvaluationFramework(
    evaluators=[SourceCoverageEvaluator(judge, verbose=False)],
    judge=judge,
)


## 2. Serial One-Call Source Coverage

Evaluate every case one at a time with `framework.evaluate`.

In [ ]:
rows = []
start = time.perf_counter()

for i, row in df.iterrows():
    case = EvaluationCase(
        context={
            "theme_business_needs": row.get("theme_businessNeeds", ""),
            "theme_description": row.get("theme_description", ""),
        },
        output={
            "epic_description": row.get("epic_description", ""),
            "epic_success_criteria": row.get("epic_successCriteria", ""),
        },
        case_id=str(row.get("epic_key", i)),
    )
    result = framework.evaluate(case)["source_coverage"]
    details = result.details or {}
    rows.append({
        "case_id": case.case_id,
        "score": result.score,
        "percentage": round(result.score * 100, 1) if result.score is not None else None,
        "label": result.label,
        "explanation": result.explanation,
        "item_count": details.get("final_item_count"),
        "judge_calls": details.get("judge_call_count"),
        "total_ms": details.get("total_ms"),
    })

wall_time_serial = time.perf_counter() - start
df_one_call_serial = pd.DataFrame(rows)

print(f"Serial wall time: {wall_time_serial:.2f}s")
display(df_one_call_serial)


## 3. Async One-Call Source Coverage

Same evaluator, all cases concurrently with `a_evaluate_many(max_concurrency=4)`.

In [ ]:
cases = []
for i, row in df.iterrows():
    cases.append(
        EvaluationCase(
            context={
                "theme_business_needs": row.get("theme_businessNeeds", ""),
                "theme_description": row.get("theme_description", ""),
            },
            output={
                "epic_description": row.get("epic_description", ""),
                "epic_success_criteria": row.get("epic_successCriteria", ""),
            },
            case_id=str(row.get("epic_key", i)),
        )
    )

start = time.perf_counter()
results_async = await framework.a_evaluate_many(cases, max_concurrency=4)
wall_time_async = time.perf_counter() - start

rows = []
for case, result_map in zip(cases, results_async):
    result = result_map["source_coverage"]
    details = result.details or {}
    rows.append({
        "case_id": case.case_id,
        "score": result.score,
        "percentage": round(result.score * 100, 1) if result.score is not None else None,
        "label": result.label,
        "explanation": result.explanation,
        "item_count": details.get("final_item_count"),
        "judge_calls": details.get("judge_call_count"),
        "total_ms": details.get("total_ms"),
    })

df_one_call_async = pd.DataFrame(rows)

print(f"Async wall time: {wall_time_async:.2f}s")
display(df_one_call_async)


## 4. DAG / Two-Stage Source Coverage

Benchmark-only. Stage 1 extracts important source items from **context only** (never
the output); Stage 2 classifies each fixed item against the output. Two judge calls
per case; score is the Python mean of covered=1.0 / partial=0.5 / missing=0.0.

In [ ]:
from idp_eval.rendering import render_value
from idp_eval.scoring import (
    calculate_coverage,
    coverage_label,
    coverage_status_from_binary,
    coverage_status_score,
)

EXTRACT_SYSTEM = (
    "Extract all materially distinct, important items from the SOURCE that a faithful "
    "representation should preserve. You are given ONLY the source; you are NOT given "
    "any generated output and must not grade anything. Consolidate duplicate or "
    "dependent statements into one item; keep independently satisfiable items "
    "separate; preserve numbers, conditions, limits, and qualifiers. There is no "
    "maximum. Return only the source item strings."
)
CLASSIFY_SYSTEM = (
    "Classify how completely the OUTPUT represents each of the fixed SOURCE ITEMS. "
    "Classify exactly those ids, once each; do not add, remove, or rewrite them. For "
    "each item return meaningfully_present (any meaningful part represented) and "
    "fully_present (full meaning incl. qualifiers represented); if not meaningful, not "
    "full. Judge meaning, not wording. Return only id and the two booleans."
)
EXTRACT_SCHEMA = {"type": "object", "properties": {"source_items": {"type": "array",
    "items": {"type": "object", "properties": {"source_item": {"type": "string"}},
              "required": ["source_item"]}}}, "required": ["source_items"]}
CLASSIFY_SCHEMA = {"type": "object", "properties": {"items": {"type": "array",
    "items": {"type": "object", "properties": {"id": {"type": "string"},
        "meaningfully_present": {"type": "boolean"}, "fully_present": {"type": "boolean"}},
        "required": ["id", "meaningfully_present", "fully_present"]}}}, "required": ["items"]}


async def extract_source_items(case):
    prompt = [{"role": "system", "content": EXTRACT_SYSTEM},
              {"role": "user", "content": "[SOURCE]\n" + render_value(case.context)}]
    resp = await asyncio.to_thread(judge.generate_object, prompt=prompt, schema=EXTRACT_SCHEMA)
    items, seen = [], set()
    for raw in resp.get("source_items", []):
        text = (raw or {}).get("source_item", "").strip()
        key = " ".join(text.lower().split())
        if text and key not in seen:
            seen.add(key)
            items.append({"id": f"s{len(items) + 1}", "source_item": text})
    return items


async def classify_source_items(items, case):
    payload = [{"id": it["id"], "source_item": it["source_item"]} for it in items]
    user = "[SOURCE ITEMS]\n" + json.dumps(payload) + "\n\n[OUTPUT]\n" + render_value(case.output)
    prompt = [{"role": "system", "content": CLASSIFY_SYSTEM}, {"role": "user", "content": user}]
    resp = await asyncio.to_thread(judge.generate_object, prompt=prompt, schema=CLASSIFY_SCHEMA)
    by_id = {a["id"]: a for a in resp.get("items", [])}
    scored = []
    for it in items:
        a = by_id[it["id"]]
        status = coverage_status_from_binary(a["meaningfully_present"], a["fully_present"])
        scored.append({"status": status, "score": coverage_status_score(status)})
    return scored


sem = asyncio.Semaphore(4)


async def run_dag(case):
    async with sem:
        t0 = time.perf_counter()
        items = await extract_source_items(case)
        t1 = time.perf_counter()
        scored = await classify_source_items(items, case) if items else []
        t2 = time.perf_counter()
    score = calculate_coverage(scored) if scored else None
    return {
        "case_id": case.case_id,
        "score": score,
        "percentage": round(score * 100, 1) if score is not None else None,
        "label": coverage_label(score) if score is not None else "not_applicable",
        "item_count": len(scored),
        "judge_calls": 2 if items else 1,
        "extract_ms": round((t1 - t0) * 1000, 1),
        "classify_ms": round((t2 - t1) * 1000, 1) if items else None,
        "total_ms": round((t2 - t0) * 1000, 1),
    }


start = time.perf_counter()
dag_rows = await asyncio.gather(*(run_dag(c) for c in cases))
wall_time_dag = time.perf_counter() - start
df_dag = pd.DataFrame(dag_rows)

print(f"DAG wall time: {wall_time_dag:.2f}s")
display(df_dag)


## 5. One-Call vs DAG Comparison

A small run-level table and a case-level score comparison. GT columns are added when
`GT_COLUMN` is present.

In [ ]:
gt = None
if GT_COLUMN in df.columns:
    gt = pd.to_numeric(df[GT_COLUMN], errors="coerce")
    gt.index = df["epic_key"].astype(str)
    gt = gt.where(gt <= 1.0, gt / 100.0)

comparison = []
for mode, wall, dfm in [
    ("one_call_serial", wall_time_serial, df_one_call_serial),
    ("one_call_async", wall_time_async, df_one_call_async),
    ("dag_async", wall_time_dag, df_dag),
]:
    entry = {
        "mode": mode,
        "wall_time_s": round(wall, 2),
        "mean_score": round(dfm["score"].mean(), 4),
        "mean_latency_ms": round(dfm["total_ms"].mean(), 1),
        "total_judge_calls": int(dfm["judge_calls"].sum()),
    }
    if gt is not None:
        err = dfm.set_index("case_id")["score"] - gt
        entry["MAE"] = round(err.abs().mean(), 4)
        entry["bias"] = round(err.mean(), 4)
    comparison.append(entry)

df_comparison = pd.DataFrame(comparison)
display(df_comparison)

df_score_comparison = pd.DataFrame({
    "case_id": df_one_call_async["case_id"],
    "one_call_score": df_one_call_async["score"].values,
    "dag_score": df_dag.set_index("case_id").loc[df_one_call_async["case_id"], "score"].values,
})
if gt is not None:
    df_score_comparison["gt_score"] = gt.loc[df_score_comparison["case_id"]].values
    df_score_comparison["one_call_abs_error"] = (df_score_comparison["one_call_score"] - df_score_comparison["gt_score"]).abs()
    df_score_comparison["dag_abs_error"] = (df_score_comparison["dag_score"] - df_score_comparison["gt_score"]).abs()
display(df_score_comparison)
